# **Kaggle – DataTops®**
Tu TA ha decidido cambiar de aires y, por eso, ha comprado una tienda de portátiles. Sin embargo, su única especialidad es Data Science, por lo que ha decidido crear un modelo de ML para establecer los mejores precios.

¿Podrías ayudar a tu profe a mejorar ese modelo?

## Aspectos importantes
- Última submission:
    - Mañana: 17 de febrero a las 5pm
    - Tarde: 19 de febrero a las 5pm
- **Enlace de la competición**: https://www.kaggle.com/t/c5cc87b50c4b4770bdc8f5acbe15577d
- **Requisito**: Estar registrado en [Kaggle](https://www.kaggle.com/)

## Métrica:
El error cuadrático medio (RMSE, por sus siglas en inglés) es una medida de la desviación estándar de los residuos (errores de predicción). Los residuos representan la diferencia entre los valores observados y los valores predichos por el modelo. El RMSE indica qué tan dispersos están estos errores: cuanto menor es el RMSE, más cercanas están las predicciones a los valores reales. En otras palabras, el RMSE mide qué tan bien se ajusta la línea de regresión a los datos.


$$ RMSE = \sqrt{\frac{1}{n}\Sigma_{i=1}^{n}{\Big(\frac{d_i -f_i}{\sigma_i}\Big)^2}}$$


## 1. Librerías

In [1]:
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
import urllib.request

## 2. Datos

In [2]:
# Para que funcione necesitas bajarte los archivos de datos de Kaggle
df = pd.read_csv("./data/train.csv")

### 2.1 Exploración de los datos

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 912 entries, 0 to 911
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   laptop_ID         912 non-null    int64  
 1   Company           912 non-null    object 
 2   Product           912 non-null    object 
 3   TypeName          912 non-null    object 
 4   Inches            912 non-null    float64
 5   ScreenResolution  912 non-null    object 
 6   Cpu               912 non-null    object 
 7   Ram               912 non-null    object 
 8   Memory            912 non-null    object 
 9   Gpu               912 non-null    object 
 10  OpSys             912 non-null    object 
 11  Weight            912 non-null    object 
 12  Price_in_euros    912 non-null    float64
dtypes: float64(2), int64(1), object(10)
memory usage: 92.8+ KB


In [4]:
df.head()

,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_in_euros
0,755,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel Core i3 6006U 2GHz,8GB,256GB SSD,Intel HD Graphics 520,Windows 10,1.86kg,539.00
1,618,Dell,Inspiron 7559,Gaming,15.6,Full HD 1920x1080,Intel Core i7 6700HQ 2.6GHz,16GB,1TB HDD,Nvidia GeForce GTX 960<U+039C>,Windows 10,2.59kg,879.01
2,909,HP,ProBook 450,Notebook,15.6,Full HD 1920x1080,Intel Core i7 7500U 2.7GHz,8GB,1TB HDD,Nvidia GeForce 930MX,Windows 10,2.04kg,900.00
3,2,Apple,Macbook Air,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,898.94
4,286,Dell,Inspiron 3567,Notebook,15.6,Full HD 1920x1080,Intel Core i3 6006U 2.0GHz,4GB,1TB HDD,AMD Radeon R5 M430,Linux,2.25kg,428.00


In [5]:
df.tail()

,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_in_euros
907,28,Dell,Inspiron 5570,Notebook,15.6,Full HD 1920x1080,Intel Core i5 8250U 1.6GHz,8GB,256GB SSD,AMD Radeon 530,Windows 10,2.2kg,800.00
908,1160,HP,Spectre Pro,2 in 1 Convertible,13.3,Full HD / Touchscreen 1920x1080,Intel Core i5 6300U 2.4GHz,8GB,256GB SSD,Intel HD Graphics 520,Windows 10,1.48kg,1629.00
909,78,Lenovo,IdeaPad 320-15IKBN,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,2TB HDD,Intel HD Graphics 620,No OS,2.2kg,519.00
910,23,HP,255 G6,Notebook,15.6,1366x768,AMD E-Series E2-9000e 1.5GHz,4GB,500GB HDD,AMD Radeon R2,No OS,1.86kg,258.00
911,229,Dell,Alienware 17,Gaming,17.3,IPS Panel Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,16GB,256GB SSD + 1TB HDD,Nvidia GeForce GTX 1060,Windows 10,4.42kg,2456.34


In [6]:
df.describe()

,laptop_ID,Inches,Price_in_euros
count,912.000000,912.000000,912.000000
mean,650.312500,14.981579,1111.724090
std,382.727748,1.436719,687.959172
min,2.000000,10.100000,174.000000
25%,324.750000,14.000000,589.000000
50%,636.500000,15.600000,978.000000
75%,982.250000,15.600000,1483.942500
max,1320.000000,18.400000,6099.000000


In [7]:
# Hacemos que nuestro índice sea el id
df.set_index("laptop_ID", inplace=True)

In [8]:
df.head()

,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_in_euros
laptop_ID,,,,,,,,,,,,
755,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel Core i3 6006U 2GHz,8GB,256GB SSD,Intel HD Graphics 520,Windows 10,1.86kg,539.00
618,Dell,Inspiron 7559,Gaming,15.6,Full HD 1920x1080,Intel Core i7 6700HQ 2.6GHz,16GB,1TB HDD,Nvidia GeForce GTX 960<U+039C>,Windows 10,2.59kg,879.01
909,HP,ProBook 450,Notebook,15.6,Full HD 1920x1080,Intel Core i7 7500U 2.7GHz,8GB,1TB HDD,Nvidia GeForce 930MX,Windows 10,2.04kg,900.00
2,Apple,Macbook Air,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,898.94
286,Dell,Inspiron 3567,Notebook,15.6,Full HD 1920x1080,Intel Core i3 6006U 2.0GHz,4GB,1TB HDD,AMD Radeon R5 M430,Linux,2.25kg,428.00


In [9]:
df_full = pd.read_csv("./data/laptops.csv", encoding='latin1')

In [10]:
for col in df_full.columns[df_full.dtypes == "object"]:
    print(f"\n{col}: {df_full[col].value_counts()}")
   


Company: Company
Dell         297
Lenovo       297
HP           274
Asus         158
Acer         103
MSI           54
Toshiba       48
Apple         21
Samsung        9
Mediacom       7
Razer          7
Microsoft      6
Vero           4
Xiaomi         4
Chuwi          3
Fujitsu        3
Google         3
LG             3
Huawei         2
Name: count, dtype: int64

Product: Product
XPS 13                                   30
Inspiron 3567                            29
250 G6                                   21
Vostro 3568                              19
Legion Y520-15IKBN                       19
                                         ..
ThinkPad L460                             1
V510-15IKB (i5-7200U/8GB/256GB/FHD/No     1
Rog GL502VS                               1
Rog GL553VE-FY052T                        1
17-ak001nv (A6-9220/4GB/500GB/Radeon      1
Name: count, Length: 618, dtype: int64

TypeName: TypeName
Notebook              727
Gaming                205
Ultrabook           

In [11]:
df_copy = df.copy()

In [12]:
# Screen resolution
features_resolution = ["Full HD", "IPS Panel", "Touchscreen", "4K Ultra HD", "Retina Display", "Quad HD+"]

for feature in features_resolution:
    df_copy[feature] = df_copy["ScreenResolution"].str.contains(feature).astype(int)
    df_copy["ScreenResolution"] = df_copy["ScreenResolution"].str.replace(feature, "")
    #df_copy.drop("ScreenResolution", axis=1, inplace=True)

df_copy["ScreenResolution"] = df_copy["ScreenResolution"].str.replace("/", "")

# Separar la resolución en dos columnas: ancho y alto
df_copy[["ScreenWidth", "ScreenHeight"]] = df_copy["ScreenResolution"].str.split("x", expand=True).astype(int)

# eliminar la columna original si ya no es necesaria
df_copy.drop("ScreenResolution", axis=1, inplace=True)
df_copy.head()

,Company,Product,TypeName,Inches,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_in_euros,Full HD,IPS Panel,Touchscreen,4K Ultra HD,Retina Display,Quad HD+,ScreenWidth,ScreenHeight
laptop_ID,,,,,,,,,,,,,,,,,,,
755,HP,250 G6,Notebook,15.6,Intel Core i3 6006U 2GHz,8GB,256GB SSD,Intel HD Graphics 520,Windows 10,1.86kg,539.00,1,0,0,0,0,0,1920,1080
618,Dell,Inspiron 7559,Gaming,15.6,Intel Core i7 6700HQ 2.6GHz,16GB,1TB HDD,Nvidia GeForce GTX 960<U+039C>,Windows 10,2.59kg,879.01,1,0,0,0,0,0,1920,1080
909,HP,ProBook 450,Notebook,15.6,Intel Core i7 7500U 2.7GHz,8GB,1TB HDD,Nvidia GeForce 930MX,Windows 10,2.04kg,900.00,1,0,0,0,0,0,1920,1080
2,Apple,Macbook Air,Ultrabook,13.3,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,898.94,0,0,0,0,0,0,1440,900
286,Dell,Inspiron 3567,Notebook,15.6,Intel Core i3 6006U 2.0GHz,4GB,1TB HDD,AMD Radeon R5 M430,Linux,2.25kg,428.00,1,0,0,0,0,0,1920,1080


In [13]:
df_copy["Ram_GB"] = df_copy["Ram"].str.replace("GB", "").astype(int)
df_copy.drop("Ram", axis=1, inplace=True)
df_copy.info()

<class 'pandas.core.frame.DataFrame'>
Index: 912 entries, 755 to 229
Data columns (total 19 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Company         912 non-null    object 
 1   Product         912 non-null    object 
 2   TypeName        912 non-null    object 
 3   Inches          912 non-null    float64
 4   Cpu             912 non-null    object 
 5   Memory          912 non-null    object 
 6   Gpu             912 non-null    object 
 7   OpSys           912 non-null    object 
 8   Weight          912 non-null    object 
 9   Price_in_euros  912 non-null    float64
 10  Full HD         912 non-null    int64  
 11  IPS Panel       912 non-null    int64  
 12  Touchscreen     912 non-null    int64  
 13  4K Ultra HD     912 non-null    int64  
 14  Retina Display  912 non-null    int64  
 15  Quad HD+        912 non-null    int64  
 16  ScreenWidth     912 non-null    int64  
 17  ScreenHeight    912 non-null    int64 

In [14]:
df_copy["Ram_GB"].value_counts()

Ram_GB
8     434
4     267
16    136
6      24
2      20
12     19
32     10
64      1
24      1
Name: count, dtype: int64

In [15]:
features_memory = ["SSD", "HDD", "Flash Storage", "Hybrid"]

# Separar la memoria en dos columnas si hay un +
df_copy[["Memory1", "Memory2"]] = df_copy["Memory"].str.split("+", expand=True).astype(str)


In [16]:
for feature in features_memory:
    df_copy[feature + "1"] = df_copy["Memory1"].apply(lambda x: x.replace(feature, "").strip() if pd.notnull(x) and feature in x else np.nan)
    df_copy[feature + "2"] = df_copy["Memory2"].apply(lambda x: x.replace(feature, "").strip() if pd.notnull(x) and feature in x else np.nan)

In [17]:
df_copy.head()

,Company,Product,TypeName,Inches,Cpu,Memory,Gpu,OpSys,Weight,Price_in_euros,...,Memory1,Memory2,SSD1,SSD2,HDD1,HDD2,Flash Storage1,Flash Storage2,Hybrid1,Hybrid2
laptop_ID,,,,,,,,,,,,,,,,,,,,,
755,HP,250 G6,Notebook,15.6,Intel Core i3 6006U 2GHz,256GB SSD,Intel HD Graphics 520,Windows 10,1.86kg,539.00,...,256GB SSD,None,256GB,NaN,NaN,NaN,NaN,NaN,NaN,NaN
618,Dell,Inspiron 7559,Gaming,15.6,Intel Core i7 6700HQ 2.6GHz,1TB HDD,Nvidia GeForce GTX 960<U+039C>,Windows 10,2.59kg,879.01,...,1TB HDD,None,NaN,NaN,1TB,NaN,NaN,NaN,NaN,NaN
909,HP,ProBook 450,Notebook,15.6,Intel Core i7 7500U 2.7GHz,1TB HDD,Nvidia GeForce 930MX,Windows 10,2.04kg,900.00,...,1TB HDD,None,NaN,NaN,1TB,NaN,NaN,NaN,NaN,NaN
2,Apple,Macbook Air,Ultrabook,13.3,Intel Core i5 1.8GHz,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,898.94,...,128GB Flash Storage,None,NaN,NaN,NaN,NaN,128GB,NaN,NaN,NaN
286,Dell,Inspiron 3567,Notebook,15.6,Intel Core i3 6006U 2.0GHz,1TB HDD,AMD Radeon R5 M430,Linux,2.25kg,428.00,...,1TB HDD,None,NaN,NaN,1TB,NaN,NaN,NaN,NaN,NaN


In [18]:
# Convertir valores de memoria a GB (soporta GB y TB) para cada columna de memoria generada
for feature in features_memory:
    for i in ["1", "2"]:
        col = feature + i
        if col in df_copy.columns:
            df_copy[col] = df_copy[col].apply(lambda x: int(x.replace("GB", "").strip()) if pd.notnull(x) and "GB" in str(x) \
            else (int(float(x.replace("TB", "").strip()) * 1024) if pd.notnull(x) and "TB" in str(x) else 0))

In [19]:
# Mostrar las filas donde la columna 'Memory' contiene un '+' (es decir, tiene dos memorias)
df_copy[df_copy["Memory"].str.contains(r"\+", na=False)].head()

,Company,Product,TypeName,Inches,Cpu,Memory,Gpu,OpSys,Weight,Price_in_euros,...,Memory1,Memory2,SSD1,SSD2,HDD1,HDD2,Flash Storage1,Flash Storage2,Hybrid1,Hybrid2
laptop_ID,,,,,,,,,,,,,,,,,,,,,
732,MSI,GL72M 7REX,Gaming,17.3,Intel Core i7 7700HQ 2.8GHz,128GB SSD + 1TB HDD,Nvidia GeForce GTX 1050 Ti,Windows 10,2.7kg,1348.48,...,128GB SSD,1TB HDD,128,0,0,1024,0,0,0,0
29,Dell,Latitude 5590,Ultrabook,15.6,Intel Core i7 8650U 1.9GHz,256GB SSD + 256GB SSD,Intel UHD Graphics 620,Windows 10,1.88kg,1298.00,...,256GB SSD,256GB SSD,256,256,0,0,0,0,0,0
358,MSI,GP72MVR 7RFX,Gaming,17.3,Intel Core i7 7700HQ 2.8GHz,128GB SSD + 1TB HDD,Nvidia GeForce GTX 1060,Windows 10,2.7kg,1409.00,...,128GB SSD,1TB HDD,128,0,0,1024,0,0,0,0
1256,MSI,GL62 6QF,Gaming,15.6,Intel Core i7 6700HQ 2.6GHz,128GB SSD + 1TB HDD,Nvidia GeForce GTX 960M,Windows 10,2.3kg,1169.00,...,128GB SSD,1TB HDD,128,0,0,1024,0,0,0,0
1277,MSI,GE62 Apache,Gaming,15.6,Intel Core i7 6700HQ 2.6GHz,128GB SSD + 1TB HDD,Nvidia GeForce GTX 960M,Windows 10,2.4kg,1229.00,...,128GB SSD,1TB HDD,128,0,0,1024,0,0,0,0


In [20]:
for feature in features_memory:
    feature1 = feature + "1"
    feature2 = feature + "2"
    feature_gb = feature + "_GB"
    df_copy[feature_gb] = df_copy[feature1] + df_copy[feature2]



In [21]:
df_copy.drop(["Memory", "Memory1", "Memory2"] + [f"{feature}{i}" for feature in features_memory for i in ["1", "2"]], axis=1, inplace=True)

In [22]:
df_copy

,Company,Product,TypeName,Inches,Cpu,Gpu,OpSys,Weight,Price_in_euros,Full HD,...,4K Ultra HD,Retina Display,Quad HD+,ScreenWidth,ScreenHeight,Ram_GB,SSD_GB,HDD_GB,Flash Storage_GB,Hybrid_GB
laptop_ID,,,,,,,,,,,,,,,,,,,,,
755,HP,250 G6,Notebook,15.6,Intel Core i3 6006U 2GHz,Intel HD Graphics 520,Windows 10,1.86kg,539.00,1,...,0,0,0,1920,1080,8,256,0,0,0
618,Dell,Inspiron 7559,Gaming,15.6,Intel Core i7 6700HQ 2.6GHz,Nvidia GeForce GTX 960<U+039C>,Windows 10,2.59kg,879.01,1,...,0,0,0,1920,1080,16,0,1024,0,0
909,HP,ProBook 450,Notebook,15.6,Intel Core i7 7500U 2.7GHz,Nvidia GeForce 930MX,Windows 10,2.04kg,900.00,1,...,0,0,0,1920,1080,8,0,1024,0,0
2,Apple,Macbook Air,Ultrabook,13.3,Intel Core i5 1.8GHz,Intel HD Graphics 6000,macOS,1.34kg,898.94,0,...,0,0,0,1440,900,8,0,0,128,0
286,Dell,Inspiron 3567,Notebook,15.6,Intel Core i3 6006U 2.0GHz,AMD Radeon R5 M430,Linux,2.25kg,428.00,1,...,0,0,0,1920,1080,4,0,1024,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28,Dell,Inspiron 5570,Notebook,15.6,Intel Core i5 8250U 1.6GHz,AMD Radeon 530,Windows 10,2.2kg,800.00,1,...,0,0,0,1920,1080,8,256,0,0,0
1160,HP,Spectre Pro,2 in 1 Convertible,13.3,Intel Core i5 6300U 2.4GHz,Intel HD Graphics 520,Windows 10,1.48kg,1629.00,1,...,0,0,0,1920,1080,8,256,0,0,0
78,Lenovo,IdeaPad 320-15IKBN,Notebook,15.6,Intel Core i5 7200U 2.5GHz,Intel HD Graphics 620,No OS,2.2kg,519.00,1,...,0,0,0,1920,1080,8,0,2048,0,0


In [23]:
# Nos aseguramos que todos los pesos son en Kg
# Listar los valores de Weight que no contienen 'Kg'
df_copy[~df_copy['Weight'].str.contains('kg', na=False)]['Weight'].unique()


array([], dtype=object)

In [24]:
df_copy["Weight_KG"] = df_copy["Weight"].str.replace("kg", "").astype(float)
df_copy.drop("Weight", axis=1, inplace=True)

In [25]:
df_copy.info()

<class 'pandas.core.frame.DataFrame'>
Index: 912 entries, 755 to 229
Data columns (total 22 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Company           912 non-null    object 
 1   Product           912 non-null    object 
 2   TypeName          912 non-null    object 
 3   Inches            912 non-null    float64
 4   Cpu               912 non-null    object 
 5   Gpu               912 non-null    object 
 6   OpSys             912 non-null    object 
 7   Price_in_euros    912 non-null    float64
 8   Full HD           912 non-null    int64  
 9   IPS Panel         912 non-null    int64  
 10  Touchscreen       912 non-null    int64  
 11  4K Ultra HD       912 non-null    int64  
 12  Retina Display    912 non-null    int64  
 13  Quad HD+          912 non-null    int64  
 14  ScreenWidth       912 non-null    int64  
 15  ScreenHeight      912 non-null    int64  
 16  Ram_GB            912 non-null    int64  
 17  

In [26]:
df_full["Cpu"].value_counts()

Cpu
Intel Core i5 7200U 2.5GHz       190
Intel Core i7 7700HQ 2.8GHz      146
Intel Core i7 7500U 2.7GHz       134
Intel Core i7 8550U 1.8GHz        73
Intel Core i5 8250U 1.6GHz        72
                                ... 
Intel Core i5 7200U 2.70GHz        1
Intel Core M M7-6Y75 1.2GHz        1
Intel Core M 6Y54 1.1GHz           1
AMD E-Series 9000 2.2GHz           1
Samsung Cortex A72&A53 2.0GHz      1
Name: count, Length: 118, dtype: int64

In [27]:
df_copy["Speed_GHz"] = df_copy["Cpu"].str.extract(r'(\d+\.?\d*)GHz').astype(float)
df_copy["Cpu_Brand"] = df_copy["Cpu"].str.extract(r'([A-Za-z]+)').iloc[:, 0]

In [28]:
df_copy["Cpu_Brand"].value_counts()

Cpu_Brand
Intel    870
AMD       42
Name: count, dtype: int64

In [29]:
df_copy.drop("Cpu", axis=1, inplace=True)

In [30]:
df_copy.info()

<class 'pandas.core.frame.DataFrame'>
Index: 912 entries, 755 to 229
Data columns (total 23 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Company           912 non-null    object 
 1   Product           912 non-null    object 
 2   TypeName          912 non-null    object 
 3   Inches            912 non-null    float64
 4   Gpu               912 non-null    object 
 5   OpSys             912 non-null    object 
 6   Price_in_euros    912 non-null    float64
 7   Full HD           912 non-null    int64  
 8   IPS Panel         912 non-null    int64  
 9   Touchscreen       912 non-null    int64  
 10  4K Ultra HD       912 non-null    int64  
 11  Retina Display    912 non-null    int64  
 12  Quad HD+          912 non-null    int64  
 13  ScreenWidth       912 non-null    int64  
 14  ScreenHeight      912 non-null    int64  
 15  Ram_GB            912 non-null    int64  
 16  SSD_GB            912 non-null    int64  
 17  

In [31]:
df_full["Gpu"].value_counts()

Gpu
Intel HD Graphics 620      281
Intel HD Graphics 520      185
Intel UHD Graphics 620      68
Nvidia GeForce GTX 1050     66
Nvidia GeForce GTX 1060     48
                          ... 
Nvidia Quadro M500M          1
AMD Radeon R7 M360           1
Nvidia Quadro M3000M         1
Nvidia GeForce 960M          1
ARM Mali T860 MP4            1
Name: count, Length: 110, dtype: int64

In [32]:
df_copy["Gpu_Brand"] = df_copy["Gpu"].str.extract(r'([A-Za-z]+)').iloc[:, 0]

In [33]:
df_copy["Gpu_Brand"].value_counts()

Gpu_Brand
Intel     509
Nvidia    284
AMD       119
Name: count, dtype: int64

In [34]:
df_copy.drop("Gpu", axis=1, inplace=True)

In [35]:
df_copy.info()

<class 'pandas.core.frame.DataFrame'>
Index: 912 entries, 755 to 229
Data columns (total 23 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Company           912 non-null    object 
 1   Product           912 non-null    object 
 2   TypeName          912 non-null    object 
 3   Inches            912 non-null    float64
 4   OpSys             912 non-null    object 
 5   Price_in_euros    912 non-null    float64
 6   Full HD           912 non-null    int64  
 7   IPS Panel         912 non-null    int64  
 8   Touchscreen       912 non-null    int64  
 9   4K Ultra HD       912 non-null    int64  
 10  Retina Display    912 non-null    int64  
 11  Quad HD+          912 non-null    int64  
 12  ScreenWidth       912 non-null    int64  
 13  ScreenHeight      912 non-null    int64  
 14  Ram_GB            912 non-null    int64  
 15  SSD_GB            912 non-null    int64  
 16  HDD_GB            912 non-null    int64  
 17  

In [36]:

df_full["Product"].value_counts()

Product
XPS 13                                   30
Inspiron 3567                            29
250 G6                                   21
Vostro 3568                              19
Legion Y520-15IKBN                       19
                                         ..
ThinkPad L460                             1
V510-15IKB (i5-7200U/8GB/256GB/FHD/No     1
Rog GL502VS                               1
Rog GL553VE-FY052T                        1
17-ak001nv (A6-9220/4GB/500GB/Radeon      1
Name: count, Length: 618, dtype: int64

In [37]:
# No aporta valor
df_copy.drop("Product", axis=1, inplace=True)

In [38]:
df_copy.info()

<class 'pandas.core.frame.DataFrame'>
Index: 912 entries, 755 to 229
Data columns (total 22 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Company           912 non-null    object 
 1   TypeName          912 non-null    object 
 2   Inches            912 non-null    float64
 3   OpSys             912 non-null    object 
 4   Price_in_euros    912 non-null    float64
 5   Full HD           912 non-null    int64  
 6   IPS Panel         912 non-null    int64  
 7   Touchscreen       912 non-null    int64  
 8   4K Ultra HD       912 non-null    int64  
 9   Retina Display    912 non-null    int64  
 10  Quad HD+          912 non-null    int64  
 11  ScreenWidth       912 non-null    int64  
 12  ScreenHeight      912 non-null    int64  
 13  Ram_GB            912 non-null    int64  
 14  SSD_GB            912 non-null    int64  
 15  HDD_GB            912 non-null    int64  
 16  Flash Storage_GB  912 non-null    int64  
 17  

In [39]:
df_full["OpSys"].value_counts()

OpSys
Windows 10      1072
No OS             66
Linux             62
Windows 7         45
Chrome OS         27
macOS             13
Mac OS X           8
Windows 10 S       8
Android            2
Name: count, dtype: int64

In [40]:
for col in df_copy.columns[df_copy.dtypes == "object"]:
    print(f"\n{col}: {df_copy[col].value_counts()}")


Company: Company
Lenovo       202
Dell         197
HP           194
Asus         121
Acer          74
MSI           37
Toshiba       34
Apple         17
Razer          6
Mediacom       6
Samsung        5
Microsoft      5
Xiaomi         3
Huawei         2
Chuwi          2
Google         2
Vero           2
Fujitsu        2
LG             1
Name: count, dtype: int64

TypeName: TypeName
Notebook              509
Gaming                143
Ultrabook             141
2 in 1 Convertible     80
Workstation            20
Netbook                19
Name: count, dtype: int64

OpSys: OpSys
Windows 10      741
Linux            48
No OS            44
Windows 7        29
Chrome OS        24
macOS            11
Windows 10 S      7
Mac OS X          6
Android           2
Name: count, dtype: int64

Cpu_Brand: Cpu_Brand
Intel    870
AMD       42
Name: count, dtype: int64

Gpu_Brand: Gpu_Brand
Intel     509
Nvidia    284
AMD       119
Name: count, dtype: int64


In [41]:
# Las compñías con menos de 10 laptops las agrupamos en "Other"

counts = df_copy["Company"].value_counts()
to_replace = counts[counts < 10].index
df_copy["Company"] = df_copy["Company"].replace(to_replace, "Other")

In [42]:
for col in df_copy.columns[df_copy.dtypes == "object"]:
    print(f"\n{col}: {df_copy[col].value_counts()}")


Company: Company
Lenovo     202
Dell       197
HP         194
Asus       121
Acer        74
MSI         37
Other       36
Toshiba     34
Apple       17
Name: count, dtype: int64

TypeName: TypeName
Notebook              509
Gaming                143
Ultrabook             141
2 in 1 Convertible     80
Workstation            20
Netbook                19
Name: count, dtype: int64

OpSys: OpSys
Windows 10      741
Linux            48
No OS            44
Windows 7        29
Chrome OS        24
macOS            11
Windows 10 S      7
Mac OS X          6
Android           2
Name: count, dtype: int64

Cpu_Brand: Cpu_Brand
Intel    870
AMD       42
Name: count, dtype: int64

Gpu_Brand: Gpu_Brand
Intel     509
Nvidia    284
AMD       119
Name: count, dtype: int64


In [43]:
# Target encoding para Company, TypeName y opSys
features_big_cat = ["Company", "TypeName", "OpSys"]
for col in features_big_cat:
    target_mean = df_copy.groupby(col)["Price_in_euros"].mean()
    df_copy[col+"_Encoded"] = df_copy[col].map(target_mean)

In [44]:
df_copy.drop(features_big_cat, axis=1, inplace=True)

In [45]:
df_copy.info()

<class 'pandas.core.frame.DataFrame'>
Index: 912 entries, 755 to 229
Data columns (total 22 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Inches            912 non-null    float64
 1   Price_in_euros    912 non-null    float64
 2   Full HD           912 non-null    int64  
 3   IPS Panel         912 non-null    int64  
 4   Touchscreen       912 non-null    int64  
 5   4K Ultra HD       912 non-null    int64  
 6   Retina Display    912 non-null    int64  
 7   Quad HD+          912 non-null    int64  
 8   ScreenWidth       912 non-null    int64  
 9   ScreenHeight      912 non-null    int64  
 10  Ram_GB            912 non-null    int64  
 11  SSD_GB            912 non-null    int64  
 12  HDD_GB            912 non-null    int64  
 13  Flash Storage_GB  912 non-null    int64  
 14  Hybrid_GB         912 non-null    int64  
 15  Weight_KG         912 non-null    float64
 16  Speed_GHz         912 non-null    float64
 17  

In [46]:
# Hacemos un one-hot encoding para las columnas de GPU y CPU
df_copy = pd.get_dummies(df_copy, columns=["Cpu_Brand", "Gpu_Brand"], drop_first=True)

In [47]:
df_copy.info()

<class 'pandas.core.frame.DataFrame'>
Index: 912 entries, 755 to 229
Data columns (total 23 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Inches            912 non-null    float64
 1   Price_in_euros    912 non-null    float64
 2   Full HD           912 non-null    int64  
 3   IPS Panel         912 non-null    int64  
 4   Touchscreen       912 non-null    int64  
 5   4K Ultra HD       912 non-null    int64  
 6   Retina Display    912 non-null    int64  
 7   Quad HD+          912 non-null    int64  
 8   ScreenWidth       912 non-null    int64  
 9   ScreenHeight      912 non-null    int64  
 10  Ram_GB            912 non-null    int64  
 11  SSD_GB            912 non-null    int64  
 12  HDD_GB            912 non-null    int64  
 13  Flash Storage_GB  912 non-null    int64  
 14  Hybrid_GB         912 non-null    int64  
 15  Weight_KG         912 non-null    float64
 16  Speed_GHz         912 non-null    float64
 17  

### 2.3 Definir X e y

In [48]:
X = df.drop(['Price_in_euros'], axis=1)
y = df['Price_in_euros'].copy()
X.shape

(912, 11)

In [49]:
y.shape

(912,)

### 2.4 Dividir X_train, X_test, y_train, y_test

In [50]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state = 42)

In [51]:
X_train

,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight
laptop_ID,,,,,,,,,,,
1118,HP,ZBook 17,Workstation,17.3,IPS Panel Full HD 1920x1080,Intel Core i7 6700HQ 2.6GHz,8GB,1TB HDD,AMD FirePro W6150M,Windows 7,3.0kg
153,Dell,Inspiron 5577,Gaming,15.6,Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,16GB,512GB SSD,Nvidia GeForce GTX 1050,Windows 10,2.56kg
275,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 2.9GHz,8GB,512GB SSD,Intel Iris Graphics 550,macOS,1.37kg
1100,HP,EliteBook 840,Notebook,14.0,Full HD 1920x1080,Intel Core i5 6200U 2.3GHz,4GB,500GB HDD,Intel HD Graphics 520,Windows 7,1.54kg
131,Dell,Inspiron 5770,Notebook,17.3,Full HD 1920x1080,Intel Core i7 8550U 1.8GHz,16GB,256GB SSD + 2TB HDD,AMD Radeon 530,Windows 10,2.8kg
...,...,...,...,...,...,...,...,...,...,...,...
578,HP,14-am079na (N3710/8GB/2TB/W10),Notebook,14.0,1366x768,Intel Pentium Quad Core N3710 1.6GHz,8GB,2TB HDD,Intel HD Graphics 405,Windows 10,1.94kg
996,Lenovo,IdeaPad 320-15ABR,Notebook,15.6,Full HD 1920x1080,AMD A12-Series 9720P 3.6GHz,6GB,256GB SSD,AMD Radeon 530,Windows 10,2.2kg
770,Dell,Latitude 7280,Ultrabook,12.5,Full HD 1920x1080,Intel Core i7 7600U 2.8GHz,16GB,256GB SSD,Intel HD Graphics 620,Windows 10,1.18kg


In [52]:
y_train

laptop_ID
1118    2899.00
153     1249.26
275     1958.90
1100    1030.99
131     1396.00
         ...   
578      389.00
996      549.00
770     1859.00
407      306.00
418     1943.00
Name: Price_in_euros, Length: 729, dtype: float64

## 3. Procesado de datos

Nuestro target es la columna `Price_in_euros`

-----------------------------------------------------------------------------------------------------------------

## 4. Modelado

### 4.1 Baseline de modelos


### 4.2 Sacar métricas, valorar los modelos

Recuerda que en la competición se va a evaluar con la métrica de ``RMSE``.

### 4.3 Optimización (up to you 🫰🏻)

-----------------------------------------------------------------

## Una vez listo el modelo, toca predecir ``test.csv``

**RECUERDA: APLICAR LAS TRANSFORMACIONES QUE HAYAS REALIZADO EN `train.csv` a `test.csv`.**


Véase:
- Estandarización/Normalización
- Eliminación de Outliers
- Eliminación de columnas
- Creación de columnas nuevas
- Gestión de valores nulos
- Y un largo etcétera de técnicas que como Data Scientist hayas considerado las mejores para tu dataset.

## 1. Carga los datos de `test.csv` para predecir.


In [53]:
X_pred = pd.read_csv("./data/test.csv")
X_pred.head()

,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight
0,209,Lenovo,Legion Y520-15IKBN,Gaming,15.6,Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,16GB,512GB SSD,Nvidia GeForce GTX 1060,No OS,2.4kg
1,1281,Acer,Aspire ES1-531,Notebook,15.6,1366x768,Intel Celeron Dual Core N3060 1.6GHz,4GB,500GB HDD,Intel HD Graphics 400,Linux,2.4kg
2,1168,Lenovo,V110-15ISK (i3-6006U/4GB/1TB/No,Notebook,15.6,1366x768,Intel Core i3 6006U 2.0GHz,4GB,1TB HDD,Intel HD Graphics 520,No OS,1.9kg
3,1231,Dell,Inspiron 7579,2 in 1 Convertible,15.6,IPS Panel Full HD / Touchscreen 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,Windows 10,2.191kg
4,1020,HP,ProBook 640,Notebook,14.0,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,4GB,256GB SSD,Intel HD Graphics 620,Windows 10,1.95kg


In [54]:
X_pred.tail()

,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight
386,820,MSI,GE72MVR 7RG,Gaming,17.3,Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,16GB,512GB SSD + 1TB HDD,Nvidia GeForce GTX 1070,Windows 10,2.9kg
387,948,Toshiba,Tecra Z40-C-12X,Notebook,14.0,IPS Panel Full HD 1920x1080,Intel Core i5 6200U 2.3GHz,4GB,128GB SSD,Intel HD Graphics 520,Windows 10,1.47kg
388,483,Dell,Precision M5520,Workstation,15.6,Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,8GB,256GB SSD,Nvidia Quadro M1200,Windows 10,1.78kg
389,1017,HP,Probook 440,Notebook,14.0,1366x768,Intel Core i5 7200U 2.5GHz,4GB,500GB HDD,Intel HD Graphics 620,Windows 10,1.64kg
390,421,Asus,ZenBook Flip,2 in 1 Convertible,13.3,IPS Panel Full HD / Touchscreen 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,Windows 10,1.27kg


In [55]:
X_pred.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 391 entries, 0 to 390
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   laptop_ID         391 non-null    int64  
 1   Company           391 non-null    object 
 2   Product           391 non-null    object 
 3   TypeName          391 non-null    object 
 4   Inches            391 non-null    float64
 5   ScreenResolution  391 non-null    object 
 6   Cpu               391 non-null    object 
 7   Ram               391 non-null    object 
 8   Memory            391 non-null    object 
 9   Gpu               391 non-null    object 
 10  OpSys             391 non-null    object 
 11  Weight            391 non-null    object 
dtypes: float64(1), int64(1), object(10)
memory usage: 36.8+ KB


 ## 2. Replicar el procesado para ``test.csv``

In [56]:
X_pred

,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight
0,209,Lenovo,Legion Y520-15IKBN,Gaming,15.6,Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,16GB,512GB SSD,Nvidia GeForce GTX 1060,No OS,2.4kg
1,1281,Acer,Aspire ES1-531,Notebook,15.6,1366x768,Intel Celeron Dual Core N3060 1.6GHz,4GB,500GB HDD,Intel HD Graphics 400,Linux,2.4kg
2,1168,Lenovo,V110-15ISK (i3-6006U/4GB/1TB/No,Notebook,15.6,1366x768,Intel Core i3 6006U 2.0GHz,4GB,1TB HDD,Intel HD Graphics 520,No OS,1.9kg
3,1231,Dell,Inspiron 7579,2 in 1 Convertible,15.6,IPS Panel Full HD / Touchscreen 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,Windows 10,2.191kg
4,1020,HP,ProBook 640,Notebook,14.0,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,4GB,256GB SSD,Intel HD Graphics 620,Windows 10,1.95kg
...,...,...,...,...,...,...,...,...,...,...,...,...
386,820,MSI,GE72MVR 7RG,Gaming,17.3,Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,16GB,512GB SSD + 1TB HDD,Nvidia GeForce GTX 1070,Windows 10,2.9kg
387,948,Toshiba,Tecra Z40-C-12X,Notebook,14.0,IPS Panel Full HD 1920x1080,Intel Core i5 6200U 2.3GHz,4GB,128GB SSD,Intel HD Graphics 520,Windows 10,1.47kg
388,483,Dell,Precision M5520,Workstation,15.6,Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,8GB,256GB SSD,Nvidia Quadro M1200,Windows 10,1.78kg
389,1017,HP,Probook 440,Notebook,14.0,1366x768,Intel Core i5 7200U 2.5GHz,4GB,500GB HDD,Intel HD Graphics 620,Windows 10,1.64kg


In [57]:
predictions_submit = model.predict(X_pred)
predictions_submit

NameError: name 'model' is not defined

**¡OJO! ¿Por qué me da error?**

IMPORTANTE:

- SI EL ARRAY CON EL QUE HICISTEIS `.fit()` ERA DE 4 COLUMNAS, PARA `.predict()` DEBEN SER LAS MISMAS
- SI AL ARRAY CON EL QUE HICISTEIS `.fit()` LO NORMALIZASTEIS, PARA `.predict()` DEBÉIS NORMALIZARLO
- TODO IGUAL SALVO **BORRAR FILAS**, EL NÚMERO DE ROWS SE DEBE MANTENER EN ESTE SET, PUES LA PREDICCIÓN DEBE TENER **391 FILAS**, SI O SI

**Entonces, si al cargar los datos de ``train.csv`` usaste `index_col=0`, ¿tendré que hacer lo también para el `test.csv`?**

In [ ]:
# ¿Qué opináis?
# ¿Sí, no?

![wow.jpeg](attachment:wow.jpeg)

## 3. **¿Qué es lo que subirás a Kaggle?**

**Para subir a Kaggle la predicción esta tendrá que tener una forma específica.**

En este caso, la **MISMA** forma que `sample_submission.csv`.

In [ ]:
sample = pd.read_csv("data/sample_submission.csv")

In [ ]:
sample.head()

In [ ]:
sample.shape

## 4. Mete tus predicciones en un dataframe llamado ``submission``.

In [ ]:
#¿Cómo creamos la submission?
submission = pd.DataFrame()

In [ ]:
submission.head()

In [ ]:
submission.shape

## 5. Pásale el CHEQUEADOR para comprobar que efectivamente está listo para subir a Kaggle.

In [ ]:
def chequeador(df_to_submit):
    """
    Esta función se asegura de que tu submission tenga la forma requerida por Kaggle.

    Si es así, se guardará el dataframe en un `csv` y estará listo para subir a Kaggle.

    Si no, LEE EL MENSAJE Y HAZLE CASO.

    Si aún no:
    - apaga tu ordenador,
    - date una vuelta,
    - enciendelo otra vez,
    - abre este notebook y
    - leelo todo de nuevo.
    Todos nos merecemos una segunda oportunidad. También tú.
    """
    if df_to_submit.shape == sample.shape:
        if df_to_submit.columns.all() == sample.columns.all():
            if df_to_submit.laptop_ID.all() == sample.laptop_ID.all():
                print("You're ready to submit!")
                df_to_submit.to_csv("submission.csv", index = False) #muy importante el index = False
                urllib.request.urlretrieve("https://www.mihaileric.com/static/evaluation-meme-e0a350f278a36346e6d46b139b1d0da0-ed51e.jpg", "gfg.png")
                img = Image.open("gfg.png")
                img.show()
            else:
                print("Check the ids and try again")
        else:
            print("Check the names of the columns and try again")
    else:
        print("Check the number of rows and/or columns and try again")
        print("\nMensaje secreto del TA: No me puedo creer que después de todo este notebook hayas hecho algún cambio en las filas de `test.csv`. Lloro.")

In [ ]:
chequeador(submission)